# Theseus 教程（中文翻译版）

- 原始英文版：`00_introduction.ipynb`
- 说明：本文件为自动翻译版本（保留代码不翻译，清空输出以减小体积）。如遇术语不一致，可优先参考英文原文。


#忒修斯

Theseus 是一个基于 PyTorch 构建的可微非线性优化库。

Theseus 的动机是机器人和计算机视觉中的问题，这些问题可以表述为可微的非线性最小二乘优化问题，例如同步定位和建图 (SLAM)、运动规划和束调整。这些问题可以大致归类为结构化学习，其中神经组件可以与已知先验进行模块化混合，以比传统方法增加价值的方式获得深度学习的好处。虽然人们对这一领域的兴趣正在迅速增加，但现有的工作是支离破碎的，并且是使用特定于应用程序的代码库构建的。 Theseus 通过为结构化学习提供一个与问题无关的平台来填补这一空白，让用户轻松地将神经网络与表示为非线性优化问题的可微分块的先验结合起来，并对这些进行端到端的训练。

本教程介绍了在Theseus 中解决此类优化问题的基本构建块；在下面的教程中，我们将展示如何将这些构建块组合在一起来解决具有各个方面且复杂性不断增加的优化问题。我们在本教程中介绍了六个概念构建块： 
* **变量：** 火炬张量的命名包装器，形成用于定义Theseus 中优化问题的基本数据类型。 （第 1 节）
* **成本函数：** 将误差项计算为一个或多个变量的函数，这些函数是Theseus 优化器要最小化的函数。 （第 2 节）
* **成本权重：**计算修改一个或多个成本函数对总体目标的贡献的权重。 （第 3 节）
* **目标：** 编译多个成本函数和权重来定义优化问题的结构。 （第 4 节）
* **优化器：** 实现可用于最小化目标的优化算法（例如，Gauss-Newton、LevenbergMarquardt）。 （第 5 节）
* **TheseusLayer：** 将目标和优化器分组，并充当火炬模块上游/下游和可微优化问题之间的接口。 （第 6 节）

## 1. 变量

Theseus 中的优化目标是 `th.Variable` 对象的函数，这些对象是不同类型（例如，2D 点、旋转组等）的 `torch.tensor` 包装器，可以选择与名称关联。在Theseus 中，我们<i>要求</i>所有变量的第一个维度是批量维度（类似于 PyTorch 模块中的约定）。我们在这里描述所有 `Variables` 共有的两个主要操作：（1）创建变量和（2）更新 `Variables`。

### 1.1 创建变量
可以使用通用 `th.Variable` 接口或通过具有自定义功能的子类来创建变量。 Theseus 应用程序中使用的许多 `Variables` 都是流形；因此，Theseus 提供了几个支持常用流形的 `Variable` 子类，例如向量、2-D/3-D 点、2-D 旋转和 2-D 刚性变换。我们在下面展示一些示例用法：

In [ ]:
import torch
import theseus as th

In [ ]:
# Create a variable with 3-D random data of batch size = 2 and name "x"
x = th.Variable(torch.randn(2, 3), name="x")
print(f"x: Named variable with 3-D data of batch size 2:\n  {x}\n")

# Create an unnamed variable. A default name will be created for it
y = th.Variable(torch.zeros(1, 1))
print(f"y: Un-named variable:\n  {y}\n")

# Create a named SE2 (2D rigid transformation) specifying data (batch_size=2)
z = th.SE2(x_y_theta=torch.zeros(2, 3).double(), name="se2_1")
print(f"z: Named SE2 variable:\n  {z}")

### 1.2 更新变量

创建变量后，可以通过 `update()` 方法更新其值。下面我们展示了一些示例以及更新变量时要避免的可能错误。

In [ ]:
# Example usage of `update`
print("Example usage of `update`: ")
print(f"  Original variable: {x}")
x.update(torch.ones(2, 3))
print(f"  Updated variable: {x}\n")

# The following inputs don't work
print("Error inputs for a Variable `update`:")
try:
    # `update` expects input tensor to respect the internal data format
    x.update(torch.zeros(2, 4))
except ValueError as e:
    print(f"  Mismatched internal data format:")
    print(f"    {e}")
try:
    # `update` expects a batch dimension
    x.update(torch.zeros(3))
except ValueError as e:
    print(f"  Missing batch dimension: ")
    print(f"    {e}\n")
    
# However the batch size can be changed via `update`
print("Change variable batch size via `update`:")
x.update(torch.ones(4, 3))
print(f"  New shape: {x.shape}")

在接下来的几节中，我们将看到 `Variable` 在Theseus 优化问题中的不同使用方式。

## 2. 成本函数

Theseus 成本函数表示一个或多个Theseus 变量的误差函数。因此，成本函数捕获了Theseus 中正在优化的核心数量。

因此，成本函数需要知道哪些变量可以优化，哪些变量不允许优化。在Theseus 中，我们通过两种变量来表示这个概念： 
* *优化变量*：Theseus 优化器可以修改这些变量以最小化目标。 
* *辅助变量*：计算目标所需的变量，但对于Theseus 优化器来说保持不变。
 
在Theseus 中，如果在创建成本函数时将 `Variable` 定义为优化变量，则 `Variable` 将成为优化变量。所有优化变量必须是`th.Manifold`的子类。

因此，需要创建一个成本函数，并声明其优化（必需）和辅助变量（可选）。成本函数提供的核心操作是使用其变量的最新值计算误差和误差的雅可比行列式。 `th.CostFunction`类是一个抽象类，要实例化它，需要实现误差计算和雅可比行列式。成本函数必须返回 `torch` 张量作为其错误。

作为一个简单的示例，我们将展示如何使用 `th.Difference` 成本函数，它是 `th.CostFunction` 的具体子类。下面，我们用两个 `Vector` 变量（一个优化变量和一个辅助变量）实例化这个成本函数。

然后，我们展示了成本函数上的一些有用的操作：成本函数如何访问其优化和辅助变量；其误差的计算，对于`th.Difference`定义为`optim_var - target` c);当底层 `Variable` 更新时，错误如何变化。最后，我们展示其雅可比矩阵的计算：这将返回一个雅可比矩阵列表，每个*优化*变量有一个条目。

In [ ]:
# Note: CostWeight is a weighting quantity required for constructing a cost function.
# We explain it in Section 3; for this example, we simply create it but we do not use it.
w1 = th.ScaleCostWeight(2.0)

# Create a Difference cost function
optim_var = th.Vector(tensor=torch.ones(1, 2), name="x1")
target = th.Vector(tensor=torch.zeros(1, 2), name="target")
cf = th.Difference(optim_var, target, w1)

# A cost function can retrieve its optimization and auxiliary variables 
print ("Retrieving the optimization and auxiliary variables from the cost function:")
print("  Optimization variables: ", list(cf.optim_vars))
print("  Auxiliary variables: ", list(cf.aux_vars))
print("")

# Cost functions compute the error using the values of the variables.
error = cf.error()
print(f"Original cost function (unweighted) error:\n  {error} of shape {error.shape}\n")

# Cost functions use the _latest_ values of the variables,
# as shown by the error values after the variable is updated.
print("Updating optimization variables by factor of 2: ")
optim_var.update(2 * torch.ones(1, 2))
print(f"  Updated variables: {optim_var}")
# Error is now twice as large as the one printed above
print(f"  Updated (unweighted) error: {cf.error()}\n")

# Compute the (unweighted) jacobians and error
# This returns a list of jacobians, with one entry per _optimization_ variable.
print("Computing cost function's (unweighted) jacobians:")
jacobians, error = cf.jacobians()  # Note cf.jacobians also returns error 
print(f"  Jacobians: {type(jacobians)} of length {len(jacobians)}")
print(f"    {jacobians[0]}")
# The i-th jacobian has shape (batch_size, cf.dim(), i-th_optim_var.dof())
print(f"    Shape of 0-th Jacobian: {jacobians[0].shape}")

在教程 3 中，我们将深入研究成本函数的内部结构，并展示如何构建自定义成本函数。

## 3. 成本权重

Theseus *成本权重* 是一种应用于成本函数的加权函数：它计算作为一个或多个变量的函数的权重，并将其应用于一个或多个成本函数的误差。因此，成本权重是修正优化问题中成本函数误差的一种方式。成本权重添加了另一层抽象，有助于在目标中的不同成本函数之间进行权衡。

`th.CostWeight` 类是抽象的，因为 `Variable` 的任何函数都可以用于创建 `CostWeight`。 Theseus 目前提供了一些具体的 `CostWeight` 子类：  
- `ScaleCostWeight`，其中加权函数是标量实数， 
- `DiagonalCostWeight`，其中加权函数是对角矩阵，
- `th.eb.GPCostWeight`，其中加权函数表示[精确稀疏高斯过程](http://roboticsproceedings.org/rss10/p01.pdf)的逆协方差函数。

`CostWeight`的主要用途是支持成本函数的`weighted_error`和`weighted_jacobians_and_error`功能；因此这些子类实现它们的（定义的）加权函数。

`CostWeight` 中使用的 `Variable` 可以是命名的，也可以是未命名的；但是，使用命名的 `Variable` 允许我们直接更新 `CostWeight` 的值；当成本权重由某些外部函数（例如 `torch.nn.Module`）计算时，这在更新 `Objective` 或 `TheseusLayer` 时特别有用。

下面我们展示了 `CostWeight` 与 `ScaleCostWeight` 类的使用示例。

In [ ]:
print("Scale cost weight creation:")
# Create a scale cost weight from a float
w1 = th.ScaleCostWeight(10.0)
# The weight is wrapped into a default variable
print(f"  w1 (default variable): {w1.scale}")

# A theseus variable can be passed directly
w2 = th.ScaleCostWeight(th.Variable(2 * torch.ones(1, 1), name="scale"))
print(f"  w2 (named variable): {w2.scale}\n")

# Weighting errors and jacobians with a ScaleCostWeight
print("Weighting errors/jacobian directly with a ScaleCostWeight:")
weighted_jacobians, weighted_error = w1.weight_jacobians_and_error(jacobians, error)
print(f"  jacobians:\n     weighted: {weighted_jacobians}\n     original: {jacobians}")
print(f"  error:\n    weighted: {weighted_error}\n    original: {error}\n")

# If the ScaleCostWeight is included in the cost function, we can directly
# use the `weight_errors` and `weight_jacobians_and_error` of the cost function.
print("Using the `weighted_error` function of the previous cost function:") 
print(f"  weighted cost function error: {cf.weighted_error()} vs unweighted error: {cf.error()}")

## 4. 目标

`th.Objective` 通过向其添加一个或多个成本函数来定义优化问题的结构，每个成本函数都具有关联的成本权重和变量。 `th.Objective` 将它们组合成一个全局误差函数，其内部结构可供Theseus 优化器使用，通过优化变量的更改来最小化全局误差。

目前，`th.Objective` 支持非线性平方和目标，其中全局误差是其每个成本函数误差的平方和，并按相应的成本权重进行加权。我们计划将来扩展到其他优化结构。创建目标的关键点是 **Theseus 假设提供的成本权重也将在最终目标中平方。** 正式而言，我们目前支持以下形式的目标

<p对齐=“中心”>
    <img src="https://raw.githubusercontent.com/facebookresearch/theseus/main/tutorials/fig/theseus_objective.png?token=ABEKAIXVTXL7BBFIRBLAEHTBWJ2R2" alt="忒修斯目标" width="250"/>
</p>

其中 **v** 表示变量集，*f*<sub>i</sub> 是成本函数误差，*w*<sub>i</sub> 是其关联的成本权重。

下面我们展示一个创建目标的简单示例。我们希望最小化以下函数 <i>(x - a)<sup>2</sup> + 4(y - b)<sup>2</sup></i>，其中 *a* 和 *b* 作为常量，*x* 和 *y* 作为变量。下面，我们首先创建（1）优化和辅助变量，（2）成本权重，（3）成本函数，（4）目标。

然后，为了评估 `Objective`，我们将使用它的 `error_metric` 函数，该函数评估误差向量的平方范数除以 2。但是，在评估它之前，我们必须至少使用一次 `Objective.update` 函数（以便正确设置内部数据结构）。一般来说，`update`函数用于轻松更改`Objective`注册的所有变量的值。该函数接收一个字典，该字典将变量名称映射到相应变量应更新到的火炬张量。

我们最终表明该函数的当前目标计算正确。 （在下一节中，我们将目标优化到最小值）

In [ ]:
# Step 1: Construct optimization and auxiliary variables.
# Construct variables of the function: these the optimization variables of the cost functions. 
x = th.Vector(1, name="x")
y = th.Vector(1, name="y")

# Construct auxiliary variables for the constants of the function.
a = th.Vector(tensor=torch.randn(1,1), name="a")
b = th.Vector(tensor=torch.randn(1,1), name="b")

# Step 2: Construct cost weights
# For w1, let's use a named variable
w1 = th.ScaleCostWeight(th.Variable(tensor=torch.ones(1, 1), name="w1_sqrt"))
w2 = th.ScaleCostWeight(2.0)  # we provide 2, as sqrt of 4 for the (y-b)^2 term

# Step 3: Construct cost functions representing each error term
# First term
cf1 = th.Difference(x, a, w1, name="term_1")
# Second term
cf2 = th.Difference(y, b, w2, name="term_2")

# Step 4: Create the objective function and add the error terms
objective = th.Objective()
objective.add(cf1)
objective.add(cf2)

# Step 5: Evaluate objective under current values
# Note this needs to be preceded by a call to `objective.update`
# Here we use the update function to set values of all variables
objective.update({"a": torch.ones(1,1), "b": 2 * torch.ones(1, 1), 
                  "x": 0.5 * torch.ones(1,1), "y": 3 * torch.ones(1, 1)})
# Weighted error should be: cost_weight * weighted_error 
print(f"Error term 1: unweighted: {cf1.error()} weighted: {cf1.weighted_error()}")
print(f"Error term 2: unweighted: {cf2.error()} weighted: {cf2.weighted_error()}")
# Objective value should be: (error1)^2 + (error2)^2 
print(f"Objective value: {objective.error_metric()}")

将成本函数添加到目标中会注册其所有优化和辅助变量（以及其成本权重的变量，如果存在）。 `th.Objective` 还检查名称没有被不同的变量或成本函数对象重载

In [ ]:
try:
    objective.add(th.Difference(y, b, w2, name="term_1"))
except ValueError as e:
    print(e)
    
try:
    obj2 = th.Objective()
    obj2.add(th.Difference(x, a, w1, name="term_1"))
    fake_x1 = th.Vector(1, name="x")
    obj2.add(th.Difference(fake_x1, b, w2, name="fake_term"))
except ValueError as e:
    print(e)

## 5. 优化器

Theseus 提供了一组线性和非线性优化器，用于最小化描述为 `th.Objective` 的问题。 
该目标可以通过调用`optimizer.optimize()`来解决，这将改变优化的值 
变量以最小化其相关目标。 `optimize` 将优化变量保留为找到的最终值， 
并返回有关优化的信息对象（其中包含最佳解决方案和优化统计信息）。

In [ ]:
# Recall that our objective is (x - a)^2 + 4 (y - b)^2
# which is minimized at x = a and y = b
# Let's start by assigning random values to them
objective.update({
    "x": torch.randn(1, 1),
    "y": torch.randn(1, 1)
})

# Now let's use the optimizer. Because this problem is minimizing a
# quadratic form, a linear optimizer can solve for the optimal solution
optimizer = th.LinearOptimizer(objective, th.CholeskyDenseSolver)
info = optimizer.optimize()

# Now let's check the values of x and y 
# Here we print only the Vectors' tensor attributes for ease of understanding
print(f"x: {x.tensor} vs a: {a.tensor}")  # Matches a = 1
print(f"y: {y.tensor} vs b: {b.tensor}")  # Matches b = 2
print(f"Objective after optimization: {objective.error_metric()}")

## 6.TheseusLayer

正如上面的警告所示，运行优化器的推荐方法是通过 `TheseusLayer`。 `TheseusLayer` 提供 `torch` 代码上游/下游与Theseus 目标和优化器之间的接口。 `forward()` 方法将 `Objective.update()` 和 `Optimizer.optimizer()` 的功能组合到单个调用中。它接收一个更新字典作为输入，并返回一个字典，其中包含优化后优化变量的火炬数据以及优化器的输出信息。

In [ ]:
layer = th.TheseusLayer(optimizer)
values, info = layer.forward({
    "x": torch.randn(1, 1),
    "y": torch.randn(1, 1),
    "a": torch.ones(1, 1),
    "b": 2 * torch.ones(1, 1),
    "w1_sqrt": torch.ones(1, 1)
})
print(f"After calling TheseusLayer's forward():")
print(f"  Values: {values}")
print(f"  Info: {info}")
print(f"  Optimized objective: {objective.error_metric()}")

`TheseusLayer` 允许反向传播，并且在语义上类似于 PyTorch 神经网络中的层。通过 `TheseusLayer` 进行反向传播可以了解问题的任何必要数量，例如成本权重、优化变量的初始值以及优化的其他参数。以下教程将说明使用 `TheseusLayer` 进行学习的几种应用。

为了区分Theseus 优化器完成的优化和Theseus 优化器外部完成的优化（例如，在学习过程中通过 PyTorch 的 autograd），我们将它们分别称为“内循环优化”和“外循环优化”。请注意，内部循环优化仅优化优化变量，外部循环优化可以优化与提供给 PyTorch autograd 优化器的选定变量相关的火炬张量。对 `TheseusLayer` `forward()` 的调用仅执行内循环优化；通常，PyTorch autograd 学习步骤将执行外循环优化。我们将在以下教程中看到这方面的示例。

在外循环期间，我们通常希望在运行内循环优化之前更新Theseus 变量；例如，设置优化变量的初始值，或者使用外循环学习到的张量更新辅助变量。我们建议通过 `TheseusLayer.forward()` 完成对Theseus 变量的此类更新。虽然变量和目标可以独立更新而无需通过 `TheseusLayer.forward()`，但遵循此约定可以明确 `TheseusLayer` 的最新输入是什么，从而有助于避免隐藏错误和不需要的行为。因此，我们建议学习期间的任何更新仅通过 `TheseusLayer` 执行。